In [5]:
import re
import sys
import pandas as pd

sys.path.append("..")
from vocab import terms

df = pd.read_csv("../data/raw/openalex_alan.csv")
df["text"] = df["title"].fillna("") + ". " + df["abstract"].fillna("")


def make_pattern(term):
    # Abbreviations like SQM, PAR, W must match exactly (capital letters).
    case_sensitive = term.isupper() or term == "lx"
    if case_sensitive:
        return re.compile(r"(?<!\w)" + re.escape(term) + r"(?!\w)")
    # Other terms ignore case and also match a plural ("moth" -> "moths").
    return re.compile(r"(?<!\w)" + re.escape(term) + r"(?:e?s)?(?!\w)", re.IGNORECASE)


def find_terms(text, term_list):
    found = [term for term in term_list if make_pattern(term).search(text)]
    return "; ".join(found)

In [10]:
kg = pd.read_csv("../kg/seed_statements.csv")

items = pd.DataFrame({"item": sorted(set(kg["subject"]) | set(kg["object"]))})

def roles(name):
    as_subject = sorted(set(kg.loc[kg["subject"] == name, "property"]))
    as_object = sorted(set(kg.loc[kg["object"] == name, "property"]))
    return pd.Series({"subject_of": ", ".join(as_subject), "object_of": ", ".join(as_object)})

items = pd.concat([items, items["item"].apply(roles)], axis=1)
print(len(items))
items

27


,item,subject_of,object_of
0,V-band,,given_in_band
1,W/m2,,reported_in
2,W/m2/nm,,reported_in
3,W/m2/sr,,reported_in
4,W/m2/sr/nm,,reported_in
5,astronomers,,used_by
6,biologists,,used_by
7,cd/m2,,reported_in
8,ecologists,,used_by
9,illuminance,"photometric_version_of, reported_in, used_by",can_derive


In [11]:
items_cls = pd.read_csv("../kg/items.csv")
print(items_cls.shape)
print(items_cls["class"].value_counts())
known = set(kg["subject"]) | set(kg["object"])
print("not found in seed_statements:", sorted(set(items_cls["item"]) - known))



(11, 2)
class
quantity         8
spectral band    2
instrument       1
Name: count, dtype: int64
not found in seed_statements: []
